# BioCulture B05 — 통합 검증과 연구 결과의 근거·한계 정리

**실제 데이터:** Dunlop Lab *E. coli* CTC, train/69 · **선행:** B01~B04 · **환경:** Python (BioImage AI), CPU

B05는 **새로운 실험이나 모델을 만들기보다** B01~B04 산출물의 *입력 일치, 데이터 품질, 결과 선택 민감도, 계보 의존성, 주장 가능 범위*를 통합 감사하는 재현성 Notebook입니다. 데이터 출처와 분석 코드, 실제 관측과 해석을 구분합니다. 본 Notebook은 실제 데이터로 사용자 PC에서 실행하기 전까지 결과 숫자를 확정하지 않습니다.

> **중요:** 한 개 mother-machine 시퀀스의 단일세포 형태·추적 자료로 OD600, CFU, 온도·배지 효과, 최적 배양 조건을 추정할 수 없습니다. 시간별 객체 수는 CFU가 아닙니다. 계보 기반 세대 구간은 관찰 주석에 기초한 후보이지 독립 확인된 분열 시간은 아닙니다.

## B05-01. 환경·프로젝트 경로와 입력 계약

Jupyter 작업 디렉터리가 `bioimage-ai` 루트 또는 `notebooks`인 경우를 모두 지원합니다. **실제 파일의 존재와 필요한 열의 계약**을 검증하고, 잘못된 경로에서 다른 CSV를 읽지 않도록 합니다.

In [ ]:
# B05-01: 라이브러리와 경로, 입력 파일 요구사항
from pathlib import Path
from zipfile import ZipFile
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

roots = [Path.cwd(), Path.cwd().parent]
ROOT = next((r.resolve() for r in roots if (r/'results'/'bioculture'/'B03'/'track_measurements.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('bioimage-ai/results/bioculture/B03/track_measurements.csv 를 찾을 수 없습니다.')
B03 = ROOT/'results'/'bioculture'/'B03'
B04 = ROOT/'results'/'bioculture'/'B04'
OUT = ROOT/'results'/'bioculture'/'B05'
OUT.mkdir(parents=True, exist_ok=True)
SOURCE_ZIP = ROOT/'data'/'bioculture'/'CTC.zip'
REQUIRED = {
 'B03': ['track_metadata.csv','frame_id_qc.csv','track_measurements.csv','growth_rate_fits.csv',
         'lineage_candidate_events.csv','growth_quality_audit.csv','lineage_generation_intervals.csv','b03_run_summary.csv'],
 'B04': ['model_comparison_by_track.csv','model_choice_counts.csv','growth_rate_distribution_summary.csv',
         'growth_filter_sensitivity.csv','time_ordered_holdout_comparison.csv','b04_run_summary.csv']
}
missing=[str((B03 if key=='B03' else B04)/name) for key, names in REQUIRED.items()
         for name in names if not ((B03 if key=='B03' else B04)/name).is_file()]
if missing: raise FileNotFoundError('필수 입력 누락:\n'+'\n'.join(missing))
print('Python:', sys.executable)
print('Project:', ROOT)
print('CTC ZIP present:', SOURCE_ZIP.is_file())
print('B03:', B03, '| B04:', B04, '| B05:', OUT)

## B05-02. B03·B04 산출물 로드와 레코드 단위 검증

측정 프레임, Cell ID, 계보표, 성장률과 모델 비교의 기본 키를 검사합니다. **주요 수치는 CSV를 직접 읽어 산출**합니다. B04의 요약 파일에 기록된 파일 개수는 당시 산출물 집계 시점의 값일 수 있으므로 실제 파일 존재 검사와 구별합니다.

In [ ]:
# B05-02: 모든 입력 데이터프레임과 기초 무결성 검사
obs=pd.read_csv(B03/'track_measurements.csv')
tracks=pd.read_csv(B03/'track_metadata.csv')
qc=pd.read_csv(B03/'frame_id_qc.csv')
fits=pd.read_csv(B03/'growth_rate_fits.csv')
events=pd.read_csv(B03/'lineage_candidate_events.csv')
quality=pd.read_csv(B03/'growth_quality_audit.csv')
intervals=pd.read_csv(B03/'lineage_generation_intervals.csv')
comparison=pd.read_csv(B04/'model_comparison_by_track.csv')
choices=pd.read_csv(B04/'model_choice_counts.csv')
dist=pd.read_csv(B04/'growth_rate_distribution_summary.csv')
sensitivity=pd.read_csv(B04/'growth_filter_sensitivity.csv')
holdout=pd.read_csv(B04/'time_ordered_holdout_comparison.csv')
summary3=pd.read_csv(B03/'b03_run_summary.csv')
summary4=pd.read_csv(B04/'b04_run_summary.csv')
for table,cols in [(obs,['cell_id','frame','time_min','area_px2','major_axis_px']),
                   (tracks,['cell_id','start','end','parent_id']),
                   (qc,['frame','missing_count','unexpected_count']),
                   (fits,['cell_id','metric','mu_per_min','r2','n_points']),
                   (comparison,['cell_id','metric','better_in_sample'])]:
    absent=set(cols)-set(table.columns)
    if absent: raise ValueError(f'입력 열 누락: {sorted(absent)}')
assert tracks.cell_id.is_unique and (tracks.cell_id>0).all()
assert not obs.duplicated(['cell_id','frame']).any()
assert not fits.duplicated(['cell_id','metric']).any()
assert not comparison.duplicated(['cell_id','metric']).any()
assert not qc.frame.duplicated().any()
assert (qc[['missing_count','unexpected_count']].fillna(0).to_numpy()==0).all(), 'TRA 정합성 실패'
assert set(obs.cell_id.astype(int)).issubset(set(tracks.cell_id.astype(int)))
assert set(obs.frame.astype(int)).issubset(set(qc.frame.astype(int)))
assert obs.groupby('cell_id')['frame'].apply(lambda x:x.is_monotonic_increasing).all(), '관측 데이터 정렬 확인 필요'
print('Frames QC:',len(qc),'| Observations:',len(obs),'| Track IDs:',tracks.cell_id.nunique())
print('Growth fits:',len(fits),'| B04 model rows:',len(comparison),'| Holdout:',len(holdout))

## B05-03. 데이터 출처와 파일 재현성 목록

원본 CTC ZIP의 존재 여부와 입력 CSV의 SHA-256을 기록합니다. **파일 해시가 동일한 것은 파일 바이트가 동일하다는 근거**이지 주석의 생물학적 정확성을 입증하는 것이 아닙니다. 시퀀스 전체를 재분할하지는 않습니다.

In [ ]:
# B05-03: 해시와 기본 계보 추적 provenance manifest 작성
files=[(B03 if stage=='B03' else B04)/f for stage, fs in REQUIRED.items() for f in fs]
if SOURCE_ZIP.is_file(): files.insert(0,SOURCE_ZIP)
def sha256(path):
    dig=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): dig.update(block)
    return dig.hexdigest()
manifest=pd.DataFrame([{'relative_path':str(f.relative_to(ROOT)), 'bytes':f.stat().st_size,
                        'sha256':sha256(f)} for f in files])
manifest.to_csv(OUT/'input_manifest_sha256.csv',index=False)
display(manifest[['relative_path','bytes']])

## B05-04. 핵심 요약 수치 재계산 및 주장 근거표

각 결과의 **표본 단위**를 구분합니다: 프레임, 추적 ID, 세포–프레임 관측, Cell ID × 지표 적합, 계보 후보. 서로 단위가 다른 건수를 합산하지 않습니다.

In [ ]:
# B05-04: 실제 테이블에서 직접 수치를 재계산합니다
counts={
 'qc_frames':len(qc), 'frames_with_id_mismatch':int(((qc.missing_count>0)|(qc.unexpected_count>0)).sum()),
 'track_metadata_rows':len(tracks),'cell_frame_observations':len(obs),'observed_cell_ids':obs.cell_id.nunique(),
 'growth_fit_rows':len(fits),'growth_fit_cell_ids':fits.cell_id.nunique(),
 'b04_model_fit_rows':len(comparison),'b04_model_cell_ids':comparison.cell_id.nunique(),
 'holdout_fit_rows':len(holdout),'lineage_candidate_intervals':len(intervals),
}
claims=pd.DataFrame([
 ('TRA 프레임 정합성', counts['frames_with_id_mismatch']==0, 'frame_id_qc.csv', '주석 ID의 구조적 일치; 추적 정확도의 독립 검증 아님'),
 ('성장률 추정 실행',counts['growth_fit_rows']>0,'growth_rate_fits.csv','개별 세포–지표별 적합; 생물학적 진실로 확정하지 않음'),
 ('원단위 성장 모델 비교',counts['b04_model_fit_rows']>0,'model_comparison_by_track.csv','동일 구간 적합오차; 새로운 환경에 대한 일반화 아님'),
 ('시간 순서 기반 holdout',len(holdout)>0,'time_ordered_holdout_comparison.csv','동일 세포의 후반 구간 예측만 평가'),
 ('계보 기반 세대 간격 후보',len(intervals)>0,'lineage_generation_intervals.csv','출생·분열 계보 기록 기반 후보; 독립 검증 아님'),
 ('배양 조건 최적화 가능',False,'CTC train/69','배지/온도 및 OD600·CFU 연결 데이터 없음'),
],columns=['claim','supported','evidence','interpretation_limit'])
metric_summary=pd.DataFrame([counts]).T.reset_index();metric_summary.columns=['metric','value']
metric_summary.to_csv(OUT/'b05_recomputed_counts.csv',index=False)
claims.to_csv(OUT/'evidence_claim_matrix.csv',index=False)
display(metric_summary);display(claims)

## B05-05. 성장률 선택 기준 민감도 재검증

B04의 민감도 CSV를 그대로 받아쓰기만 하지 않고 B03 적합표에서 직접 재계산하여 대조합니다. **관측점 수와 R² 기준을 바꾸면 분석대상과 중앙값이 달라지는 것**을 확인합니다.

In [ ]:
# B05-05: B04 민감도 CSV vs B03 직접 재계산
recomputed=[]
for n_min in sorted(sensitivity.min_points.unique()):
    for r_min in sorted(sensitivity.min_r2.unique()):
        sub=fits.loc[(fits.n_points>=n_min)&(fits.r2>=r_min)&np.isfinite(fits.mu_per_min)&(fits.mu_per_min>0)]
        for metric in sorted(sensitivity.metric.unique()):
            values=sub.loc[sub.metric.eq(metric),'mu_per_min']
            recomputed.append({'metric':metric,'min_points':n_min,'min_r2':r_min,
                               'n_tracks_check':len(values),'median_mu_check':values.median()})
check=pd.DataFrame(recomputed)
merged=sensitivity.merge(check,on=['metric','min_points','min_r2'],how='outer',validate='one_to_one',indicator=True)
assert merged['_merge'].eq('both').all(),'민감도 조합 불일치'
assert (merged.n_tracks==merged.n_tracks_check).all(),'대상 셀 수 불일치'
assert np.allclose(merged.median_mu_per_min.to_numpy(dtype=float), merged.median_mu_check.to_numpy(dtype=float),equal_nan=True),'중앙값 불일치'
merged.drop(columns=['_merge']).to_csv(OUT/'filter_sensitivity_crosscheck.csv',index=False)
focus=merged.loc[(merged.metric=='major_axis_px')&np.isclose(merged.min_r2,0.70)].sort_values('min_points')
display(focus[['min_points','n_tracks','median_mu_per_min']])
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(focus.min_points,focus.median_mu_per_min,marker='o')
ax.set(xlabel='Minimum observations (R² >= 0.70)',ylabel='Median log-growth slope (/min)',title='Sensitivity to minimum track length')
ax.grid(alpha=.25);fig.tight_layout();fig.savefig(OUT/'growth_threshold_sensitivity.png',dpi=160);plt.show()

## B05-06. 동일 구간 적합과 시간 순서 예측 평가를 분리하여 기록

동일 세포–지표에 대한 **in-sample 모델 우세 횟수**와 **holdout 우세 횟수**는 표본 조건이 다를 수 있으므로 직접 비율 비교에 유의해야 합니다. 모델 간 차이가 관찰되더라도 기계적 우열·인과효과로 단정하지 않습니다.

In [ ]:
# B05-06: 두 종류의 모델 비교 결과를 구분하여 보고
assert 'better_holdout' in holdout.columns
ins= comparison.groupby(['metric','better_in_sample']).size().reset_index(name='n')
ins.insert(0,'evaluation','in_sample')
oos=holdout.groupby(['metric','better_holdout']).size().reset_index(name='n')
oos=oos.rename(columns={'better_holdout':'better_in_sample'});oos.insert(0,'evaluation','same_track_future_holdout')
model_audit=pd.concat([ins,oos],ignore_index=True)
model_audit=model_audit.rename(columns={'better_in_sample':'winning_model'})
model_audit.to_csv(OUT/'model_evaluation_scope.csv',index=False)
display(model_audit.sort_values(['metric','evaluation','winning_model']))

## B05-07. 계보 의존성과 관측 종료 구분

한 계보에서 부모와 자식의 성장률은 독립 표본이 아닐 수 있습니다. Cell ID 수를 그대로 생물학적 독립 반복 수로 취급하면 불확실성을 과소평가할 수 있습니다. **관측 구간 종결=분열**이라는 해석도 금지합니다.

In [ ]:
# B05-07: 계보 구조 및 시퀀스 경계 사례 요약
assert {'cell_id','start','end','parent_id'}.issubset(tracks.columns)
start_frame=int(qc.frame.min());end_frame=int(qc.frame.max())
lineage_audit=pd.DataFrame([{
 'sequence_start_frame':start_frame,'sequence_end_frame':end_frame,
 'ids_touching_start':int((tracks.start==start_frame).sum()),
 'ids_touching_end':int((tracks.end==end_frame).sum()),
 'ids_with_annotated_parent':int((tracks.parent_id!=0).sum()),
 'unique_parent_ids':int(tracks.loc[tracks.parent_id!=0,'parent_id'].nunique()),
 'generation_interval_candidates':len(intervals)
}])
assert set(tracks.loc[tracks.parent_id!=0,'parent_id'].astype(int)).issubset(set(tracks.cell_id.astype(int)))
lineage_audit.to_csv(OUT/'lineage_boundary_audit.csv',index=False)
display(lineage_audit)
print('해석 제한: 시작/종료 프레임을 포함하는 추적과 계보 후보는 독립 관찰된 세대시간과 다릅니다.')

## B05-08. 과학적 결론·한계 및 향후 데이터 요구조건

아래 결론은 **실측 CTC 시퀀스 분석의 범위**에 한정됩니다. OD600/CFU·배양 최적화를 위한 추가 데이터는 별도의 수집·연결 검증을 거쳐야 합니다.

In [ ]:
# B05-08: 연구 요약을 기계판독 가능 파일로 기록
interpretation={
 'scope':'Single CTC train/69 mother-machine sequence; morphology and tracking',
 'supported':[ 'frame-index and annotated tracking ID structural consistency',
               'cell-wise morphology time series and conditional growth fits',
               'exploratory model comparison and threshold sensitivity',
               'annotated lineage generation-interval candidates'],
 'not_supported':[ 'OD600 or CFU estimates', 'causal temperature or medium effect',
                   'optimized culture conditions','population-generalizable confidence intervals',
                   'independently confirmed biological division times'],
 'next_steps':[ 'human visual QC of representative track masks and segmentation boundaries',
                'lineage-aware statistical sensitivity analysis if additional independent sequences exist',
                'obtain independent culture-condition datasets before optimization']
}
(OUT/'b05_scientific_interpretation.json').write_text(json.dumps(interpretation,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(interpretation,ensure_ascii=False,indent=2))

## B05-09. 출력물 검증과 실행 종료 보고

처리 결과의 필수 파일과 데이터 행 수를 확인합니다. 결과 파일을 GitHub에 게시할 때는 원본 데이터의 재배포 조건을 따로 확인해야 합니다.

In [ ]:
# B05-09: 필수 산출물을 검사하고 실행 종료 상태를 출력
required_outputs=['input_manifest_sha256.csv','b05_recomputed_counts.csv','evidence_claim_matrix.csv',
 'filter_sensitivity_crosscheck.csv','growth_threshold_sensitivity.png',
 'model_evaluation_scope.csv','lineage_boundary_audit.csv','b05_scientific_interpretation.json']
not_found=[name for name in required_outputs if not (OUT/name).is_file()]
assert not not_found, f'출력 누락: {not_found}'
assert len(qc)>0 and len(obs)>0 and len(comparison)>0
print('B05 completed; validated output files:',len(required_outputs))
for name in required_outputs:print(' -',name)
print('B05 결과는 검증 요약이며, 새로운 배양 조건이나 실험 결과는 생성하지 않았습니다.')

## 학습 확인 문제 — 해답은 B05 Word 실습 해설서에서 제공

1. `observations`, `track IDs`, `model fit rows`가 서로 다른 이유를 설명하십시오.
2. 최소 관측점 기준을 4·6·8로 바꿀 때 중앙 성장률이 달라지는 이유를 두 가지 제시하십시오.
3. 동일 세포에서 모델 적합오차와 후반 시점 예측오차는 무엇이 다릅니까?
4. `man_track.txt`의 부모–자식 ID가 일치한다는 사실만으로 정확한 세대시간이 확정되지 않는 이유는 무엇입니까?
5. OD600, CFU, 배양 조건 최적화를 실제 데이터로 연구하려면 어떤 메타데이터와 측정값이 추가로 필요합니까?

**다음 단계:** 실제 CTC 실행 완료 `.ipynb`를 저장하여 검증한 후 B05 실습 Word 해설서를 작성합니다. **BioCulture의 종료 범위(B05로 완료 또는 조건별 데이터로 확장)는 결과 검증 후 결정**합니다.